# Fusão de Bases de Dados (Merge) - fNIRS e Fisiologia Periférica | Tarefa: Stroop
Este script unifica as métricas neurofisiológicas (Betas do GLM para os blocos Congruente, Incongruente e a diferença Incongruente > Congruente) com as variáveis autonómicas (GSR e Frequência Cardíaca) extraídas da tarefa Stroop.
A fusão usa a chave `ID` num *Inner Join*, garantindo que apenas sujeitos com sinal cerebral e corporal viável transitam para a base de dados final (Pronta para SPSS / R).

In [1]:
import os
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. Configuração de Caminhos
# ==========================================
BASE_DIR = r"C:\Users\guilh\Desktop\OneDrive - Universidade de Coimbra\PhD_Project_ARDITI_Guilherme Saldanha\Study2_Data Collection Files_PhD_Guilherme Saldanha_UC-UMa-ITI"
PASTA_MNE_NIRS = os.path.join(BASE_DIR, "derivatives", "mne-nirs")

FICHEIRO_FNIRS = "dataset_mestre_fnirs_STROOP.csv"
FICHEIRO_FISIO = "dataset_mestre_fisio_STROOP.csv"

caminho_fnirs = os.path.join(PASTA_MNE_NIRS, FICHEIRO_FNIRS)
caminho_fisio = os.path.join(PASTA_MNE_NIRS, FICHEIRO_FISIO)

print("A iniciar processo de fusão (Merge) do Stroop...")

try:
    # ==========================================
    # 2. Leitura dos Dados
    # ==========================================
    df_fnirs = pd.read_csv(caminho_fnirs)
    
    # engine python e sep=None para detectar automaticamente o separador
    df_fisio = pd.read_csv(caminho_fisio, sep=None, engine='python')
    
    # Limpar espaços em branco na chave ID
    df_fnirs['ID'] = df_fnirs['ID'].str.strip()
    df_fisio['ID'] = df_fisio['ID'].str.strip()

    # Remover a coluna Tarefa da Fisiologia se existir
    if 'Tarefa' in df_fisio.columns:
        df_fisio = df_fisio.drop(columns=['Tarefa'])

    # ==========================================
    # 3. Merge (Inner Join)
    # ==========================================
    print("A cruzar os dados usando a coluna 'ID'...")
    df_master = pd.merge(df_fnirs, df_fisio, on='ID', how='inner')
    
    # ==========================================
    # 4. Ordenação das Colunas
    # ==========================================
    colunas_base = ['ID', 'Tarefa', 'Canais_Rejeitados_SCI']
    colunas_fisiologia = [c for c in df_fisio.columns if c != 'ID']
    colunas_hbo = [col for col in df_fnirs.columns if col.startswith('HbO_')]
    
    todas_colunas = colunas_base + colunas_fisiologia + colunas_hbo
    df_master = df_master[todas_colunas]

    # ==========================================
    # 5. Exportação Final
    # ==========================================
    caminho_exportacao = os.path.join(PASTA_MNE_NIRS, "dataset_FINAL_SPA_fNIRS_Stroop.csv")
    df_master.to_csv(caminho_exportacao, index=False)
    
    print("\n" + "="*50)
    print("FUSÃO CONCLUÍDA COM SUCESSO! (STROOP)")
    print("="*50)
    print(f"-> Número de sujeitos integrados: {df_master.shape[0]}")
    print(f"-> Total de variáveis na tabela: {df_master.shape[1]}")
    print(f"-> Ficheiro guardado em:\n{caminho_exportacao}\n")
    
    print("Amostra das primeiras 2 linhas e 7 colunas iniciais:")
    display(df_master.iloc[:2, :7])

except Exception as e:
    print(f"ERRO durante a leitura/fusão: {e}")

A iniciar processo de fusão (Merge) do Stroop...
A cruzar os dados usando a coluna 'ID'...

FUSÃO CONCLUÍDA COM SUCESSO! (STROOP)
-> Número de sujeitos integrados: 53
-> Total de variáveis na tabela: 131
-> Ficheiro guardado em:
C:\Users\guilh\Desktop\OneDrive - Universidade de Coimbra\PhD_Project_ARDITI_Guilherme Saldanha\Study2_Data Collection Files_PhD_Guilherme Saldanha_UC-UMa-ITI\derivatives\mne-nirs\dataset_FINAL_SPA_fNIRS_Stroop.csv

Amostra das primeiras 2 linhas e 7 colunas iniciais:


,ID,Tarefa,Canais_Rejeitados_SCI,GSR_Media_Con,GSR_Amplitude_Con,GSR_Media_Inc,GSR_Amplitude_Inc
0,sub-001,Stroop,18,-0.047542,0.034212,-0.065105,0.022514
1,sub-002,Stroop,16,0.688339,1.324277,0.582108,1.098462
